In [ ]:
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
plt.style.use('seaborn-v0_8-colorblind')

In [ ]:
import yaml

base_path = Path('../..').resolve()
sys.path.append(str(base_path))
from helpers import singlecell_utils

with open(base_path / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

# Read PsychAD subclass - class mapper

In [ ]:
# Gene metadata required
FULL_H5AD_PATH = '/sc/arion/projects/psychAD/NPS-AD/freeze2_proc/240124_PsychAD_freeze3_FULL_clean.h5ad'
psyad_full = singlecell_utils.read_everything_but_X(FULL_H5AD_PATH)

In [ ]:
df_subclass_to_class = psyad_full.obs.groupby('subclass', observed=False)['class'].first().reset_index()
subclass_to_class = dict(zip(df_subclass_to_class['subclass'], df_subclass_to_class['class']))
subclass_to_class

# Read AD multiome cell type annotation per each barcode

In [ ]:
CELL_BARCODE_PATH = '/sc/arion/projects/CommonMind/roussp01a/snmulti/step3/files/cell_anno_05172024_filter.csv'
df_cb = pd.read_csv(CELL_BARCODE_PATH, index_col=0)
df_cb

# Read transcript read counts from Long-read seq

In [ ]:
H5AD_PATH = '/sc/arion/projects/CommonMind/roussp01a/MAS/rerun2/cPBi_counts_celltype.h5ad'
dat = ad.read_h5ad(H5AD_PATH)

In [ ]:
#GABA = Inhibitory
#GLU = Excitatory

In [ ]:
print(dat.obs['cell_type'].cat.categories)
print(dat.obs['cell_type2'].cat.categories)

In [ ]:
# Most of long-read cells should be included in the entire AD multiome cell type annotation

dat.obs['Multiome_class'] = df_cb['class']
dat.obs['subclass'] = df_cb['subclass']
dat.obs['class'] = dat.obs['subclass'].map(subclass_to_class)

dat.obs[dat.obs['subclass'].isnull()]

In [ ]:
# Vascular of AD multiome is Endo + Mural of PsychAD. I will use PsychAD class

dat.obs[dat.obs['class']!=dat.obs['Multiome_class']][['Multiome_class', 'class', 'subclass']].drop_duplicates()

In [ ]:
Counter(dat.obs['class'])

In [ ]:
dat = dat[~dat.obs['class'].isnull(), :]

In [ ]:
df_psb = singlecell_utils.get_pseudobulk(dat, 'class')
df_psb.head()

In [ ]:
print('Cell types has different read counts, so CPM is required for the PAS selection')

fig, ax = plt.subplots()

for col in df_psb.sum().sort_values().index:
    se_values = np.log2(df_psb[col].sort_values()+0.5) 
    se_quantiles = se_values.rank() / len(se_values)
    ax.plot(se_values, se_quantiles, label=col)

ax.set_title('MAS-seq, cell_type')
ax.set_ylabel('Cumulative fraction')
ax.set_xlabel('log2(count + 0.5)')
ax.legend(loc=(1.01, 0.0))
plt.show()


In [ ]:
df_psb = df_psb.astype(int)
df_psb_cpm = (df_psb+0.5) / ((df_psb+0.5).sum() / 10**6)
for col in df_psb:
    df_psb['cpm_' + col] = df_psb_cpm[col]

df_psb

In [ ]:
print('Cell types has different read counts, so CPM is required for the PAS selection')

fig, ax = plt.subplots()

for col in df_psb_cpm.sum().sort_values().index:
    se_values = np.log2(df_psb_cpm[col].sort_values()+0.5) 
    se_quantiles = se_values.rank() / len(se_values)
    ax.plot(se_values, se_quantiles, label=col)

ax.set_title('MAS-seq, cell_type')
ax.set_ylabel('Cumulative fraction')
ax.set_xlabel('log2(count + 0.5)')
ax.legend(loc=(1.01, 0.0))
plt.show()


In [ ]:
df_psb.sum()

In [ ]:
print(dat.var['total_counts'].min(), dat.var['total_counts'].max())

fig = plt.figure()
ax = fig.add_subplot(111)
np.log2(dat.var['total_counts']).hist(bins=40, ax=ax)
ax.set_title('cPBi_counts_celltype.h5ad')
ax.set_xlabel('log2 raw read count')
ax.set_ylabel('Number of transcript')
plt.show()

In [ ]:
print(dat.var['n_cells_by_counts'].min(), dat.var['n_cells_by_counts'].max())

fig = plt.figure()
ax = fig.add_subplot(111)
np.log2(dat.var['n_cells_by_counts']).hist(bins=40, ax=ax)
ax.set_title('cPBi_counts_celltype.h5ad')
ax.set_xlabel('log2 n_cells_by_counts')
ax.set_ylabel('Number of transcript')
plt.show()

# Read talon annotation from long-read sequencing

In [ ]:
df_talon = pd.read_table('/sc/arion/projects/CommonMind/roussp01a/MAS/rerun2/talon/combined_talon_observedOnly.gtf',
                       header=None, index_col=False, dtype={0: 'category'})
df_talon

In [ ]:
df_talon_tx = df_talon[df_talon[2]=='transcript']
df_talon_pos = df_talon_tx[df_talon_tx[6]=='+'].copy()
df_talon_neg = df_talon_tx[df_talon_tx[6]=='-'].copy()

In [ ]:
# Get 3p end
df_talon_pos[3] = df_talon_pos[4] - 1
df_talon_neg[4] = df_talon_neg[3] + 1

In [ ]:
df_talon_3p = pd.concat([df_talon_pos, df_talon_neg]).sort_values([0, 3, 4])
# Replace chromosom names
df_talon_3p[0] = df_talon_3p[0].str.replace('chrM', 'MT').str.replace('chr', '')

In [ ]:
Counter(df_talon_3p[0])

In [ ]:
df_talon_3p[8].iloc[0]

In [ ]:
def parse_desc(desc, second_delim):
    return dict(field.strip().replace('"', '').split(second_delim) for field in desc.removesuffix(';').split(';'))

df_talon_desc = pd.DataFrame(list(df_talon_3p[8].apply(parse_desc, args=' ')), index=df_talon_3p.index).convert_dtypes()
df_talon_desc['gene_index'] = df_talon_desc['gene_name'] + ':' + df_talon_desc['transcript_id']
df_talon_desc.head()

In [ ]:
df_talon_3p['gene_index'] = df_talon_desc['gene_index']
df_talon_3p.head()

# Merge read counts and talon annotation

In [ ]:
df_3p_counts = pd.merge(df_talon_3p[[0, 3, 4, 6, 'gene_index']], df_psb, left_on='gene_index', right_index=True, how='inner')
df_3p_counts.rename(columns={0: 'chr', 3: 'start', 4:'end', 6: 'strand'}, inplace=True)
df_3p_counts['gene_name'] = df_3p_counts['gene_index'].str.split(':').apply(lambda x: x[0])
df_3p_counts

# Expand 3' ends of transcripts

In [ ]:
df_3p_counts['start-50'] = df_3p_counts['start'] - 50
df_3p_counts['end+50'] = df_3p_counts['end'] + 50

df_3p_counts['start-70'] = df_3p_counts['start'] - 70
df_3p_counts['end+70'] = df_3p_counts['end'] + 70

In [ ]:
df_3p_counts.head()

In [ ]:
df_3p_counts.to_pickle('long-read_3p_and_psb_cnt_cpm.pkl')
df_3p_counts.to_csv('long-read_3p_and_psb_cnt_cpm.tsv.gz', sep='\t', compression='gzip')